# NaNLM: Qwen 7B teacher on AMD Radeon Cloud

Use **AMD OneClick Base** (ROCm 7.2, PyTorch 2.9.1, Python 3.12). Do **not** run
`pip install torch`; the AMD docs warn that it replaces the ROCm build with a
CUDA build. Install only `transformers`, `accelerate`, `sentencepiece`, and
`huggingface_hub`.

The goal is one cached teacher pass, not 7B fine-tuning.


In [ ]:
%%bash
rocm-smi
/opt/venv/bin/python - <<'PY'
import torch
print('torch', torch.__version__)
print('hip', torch.version.hip)
print('cuda_available', torch.cuda.is_available())
print('gpu', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
PY

In [ ]:
%%bash
/opt/venv/bin/pip install -q transformers accelerate sentencepiece huggingface_hub

In [ ]:
%%bash
cd /workspace
test -d NaNLM || git clone https://github.com/shemshallah/NaNLM.git
cd /workspace/NaNLM
git rev-parse --short HEAD

In [ ]:
%%bash
mkdir -p /workspace/models
if [ ! -d /workspace/models/qwen2.5-7b-instruct ]; then
  /opt/venv/bin/huggingface-cli download Qwen/Qwen2.5-7B-Instruct \
    --local-dir /workspace/models/qwen2.5-7b-instruct
fi
du -sh /workspace/models/qwen2.5-7b-instruct

In [ ]:
%%bash
cd /workspace/NaNLM
mkdir -p build
/opt/venv/bin/python tools/hf_teacher.py \
  --model /workspace/models/qwen2.5-7b-instruct \
  --corpus data/alice_excerpt.txt \
  --out build/qwen7b_alice.ntr \
  --card build/qwen7b_alice.card.json \
  --max-tokens 2048 \
  --top-k 16 \
  --chunk 128 \
  --device cuda \
  --dtype float16

In [ ]:
import json
from pathlib import Path
card = json.loads(Path('/workspace/NaNLM/build/qwen7b_alice.card.json').read_text())
print(json.dumps(card, indent=2, sort_keys=True))


## Compile the 8-slot and 3-slot students

If the teacher card does not beat the 3-gram BPC, stop; the run is a negative
teacher result, not a student result.


In [ ]:
%%bash
cd /workspace/NaNLM
make build/nanlm_teacher build/nanlm_slot8_train build/nanlm_teacher_slot build/nanlm_asm build/nanlm_vm
./build/nanlm_teacher stats build/qwen7b_alice.ntr

In [ ]:
%%bash
cd /workspace/NaNLM
TEACHER=build/qwen7b_alice.ntr \
CORPUS=data/alice_excerpt.txt \
MAX_KEYS=2048 \
sh tools/build_slot8.sh

In [ ]:
%%bash
cd /workspace/NaNLM
/opt/venv/bin/python - <<'PY'
import subprocess
subprocess.run([
  './build/nanlm_slot8_train','--teacher','build/qwen7b_alice.ntr',
  '--corpus','data/alice_excerpt.txt','--max-keys','2048',
  '--eval-log','build/qwen7b_slot8.tsv','--out','build/qwen7b_slot8.nas',
  '--expected','build/qwen7b_slot8.expected',
  '--manifest','build/qwen7b_slot8.manifest'], check=True)
subprocess.run([
  './build/nanlm_teacher_slot','--teacher','build/qwen7b_alice.ntr',
  '--corpus','data/alice_excerpt.txt',
  '--eval-log','build/qwen7b_slot3.tsv','--out','build/qwen7b_slot3.nas',
  '--expected','build/qwen7b_slot3.expected',
  '--manifest','build/qwen7b_slot3.manifest'], check=True)
PY

In [ ]:
%%bash
cd /workspace/NaNLM
/opt/venv/bin/python experiments/eval_stream.py \
  --run slot8=build/qwen7b_slot8.tsv=build/qwen7b_slot8.nas \
  --run slot3=build/qwen7b_slot3.tsv=build/qwen7b_slot3.nas \
  --boot 5000

In [ ]:
%%bash
cd /workspace/NaNLM
tar czf /workspace/qwen7b_run.tgz \
  build/qwen7b_alice.ntr build/qwen7b_alice.card.json \
  build/qwen7b_slot8.tsv build/qwen7b_slot3.tsv \
  build/qwen7b_slot8.nas build/qwen7b_slot8.expected \
  build/qwen7b_slot8.manifest build/qwen7b_slot3.nas \
  build/qwen7b_slot3.expected build/qwen7b_slot3.manifest
ls -lh /workspace/qwen7b_run.tgz
sha256sum build/qwen7b_alice.ntr build/qwen7b_alice.card.json

## Next

If the Alice gate passes, run a pinned PG-19 book or `build/continuous/corpus.txt`
with `--max-tokens 8192`. Keep the model and `/workspace` persistent so the
second run does not re-download 15 GB. Upload only `qwen7b_run.tgz`; never the
7B weights.
